In [ ]:
import os
import ast
import pandas as pd
import torch
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from peft import PeftModel

# Set your paths
DATA_DIR = Path("data")
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# REPLACE this with the actual path to your saved checkpoint from training
CHECKPOINT_DIR = "logs/dora_vision_ckpt_epoch5_20260430_124557"

In [ ]:
test_df = pd.read_csv(DATA_DIR / "test.csv")
test_df["choices"] = test_df["choices"].apply(ast.literal_eval)

print(f"Test examples loaded: {len(test_df)}")

In [ ]:
def build_nli_prompt(row, choice_text, max_context_chars=2000):
    text = ""
    if pd.notna(row.get("lecture", None)) and row["lecture"]:
        text += f"Background:\n{row['lecture'][:max_context_chars]}\n\n"
    if pd.notna(row.get("hint", None)) and row["hint"]:
        text += f"Passage:\n{row['hint'][:max_context_chars]}\n\n"
    
    text += f"Question: {row['question']}\n"
    text += f"Proposed answer: {choice_text}\n"
    text += "Is this the correct answer? Yes or No."

    prompt = f"<|im_start|>User:<image>{text}<end_of_utterance>\nAssistant:"
    return prompt

In [ ]:
print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

processor.image_processor.do_image_splitting = False
processor.image_processor.size = {"longest_edge": 512}
processor.image_processor.max_image_size = {"longest_edge": 512}

# Get IDs for margin calculation
yes_token = processor.tokenizer("Yes", add_special_tokens=False)["input_ids"][0]
no_token  = processor.tokenizer("No",  add_special_tokens=False)["input_ids"][0]

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Loading base model...")
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="cuda:0",
    low_cpu_mem_usage=True,
)
base_model.config.use_cache = False

print(f"Attaching PEFT checkpoint from {CHECKPOINT_DIR}...")
model = PeftModel.from_pretrained(base_model, CHECKPOINT_DIR)
model.eval()

print("Model is ready for inference!")

In [ ]:
def score_choices_nli(model, processor, image, row, choices, max_context_chars=2000):
    scores = []
    for choice in choices:
        prompt = build_nli_prompt(row, choice, max_context_chars=max_context_chars)
        
        inputs = processor(
            text=[prompt], images=[image],
            return_tensors="pt", padding=True
        )
        inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            logits  = outputs.logits[:, -1, :]

        # Confidence margin: P(Yes) - P(No)
        score = logits[0, yes_token].item() - logits[0, no_token].item()
        scores.append(score)

        del outputs, logits, inputs
        torch.cuda.empty_cache()

    # Return the index (0, 1, 2, etc.) of the choice with the highest margin
    return int(torch.tensor(scores).argmax().item())

In [ ]:
test_results = []

for idx in tqdm(range(len(test_df)), desc="Test inference"):
    row   = test_df.iloc[idx]
    image = Image.open(DATA_DIR / row["image_path"]).convert("RGB")
    
    pred  = score_choices_nli(model, processor, image, row, row["choices"])
    test_results.append({"id": row["id"], "answer": pred})

submission_df = pd.DataFrame(test_results)
submission_df.to_csv("submission.csv", index=False)

print(f"Saved submission.csv successfully!")
print(f"Shape: {submission_df.shape}")